# 03 - Image Exploration (EDA)

Read-only exploration of raw FITS images from `dataset/raw/` using `dataset/metadata/manifest.csv` and `dataset/metadata/image_stats.csv`. No raw data is modified.

In [ ]:
from pathlib import Path
import csv
import random
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

ROOT = Path(".")
RAW = ROOT / "dataset" / "raw"
MANIFEST = ROOT / "dataset" / "metadata" / "manifest.csv"
STATS = ROOT / "dataset" / "metadata" / "image_stats.csv"

with MANIFEST.open(newline="", encoding="utf-8") as f:
    manifest = list(csv.DictReader(f))
by_id = {r["sample_id"]: r for r in manifest}
print(len(manifest), "samples")

In [ ]:
def display_stretch(data, p_low=1.0, p_high=99.5):
    """DISPLAY ONLY: not training preprocessing.

    Percentile clipping + asinh stretch for visualization. Works on a copy of
    finite values; non-finite pixels are left as NaN in the display array.
    This function must never write data back anywhere.
    """
    arr = np.asarray(data, dtype=np.float64)
    finite = np.isfinite(arr)
    if not np.any(finite):
        return np.zeros_like(arr, dtype=np.float32)
    vals = arr[finite]
    lo = np.percentile(vals, p_low)
    hi = np.percentile(vals, p_high)
    if not np.isfinite(lo):
        lo = np.nanmin(vals)
    if not np.isfinite(hi) or hi <= lo:
        hi = lo + 1.0
    clipped = np.clip(arr, lo, hi)
    # scale to [0,1] then asinh for dynamic range
    scaled = (clipped - lo) / (hi - lo)
    stretched = np.arcsinh(10.0 * scaled) / np.arcsinh(10.0)
    out = stretched.astype(np.float32)
    out[~finite] = np.nan
    return out

## Gallery: 6 random samples (seed 42), all 5 HDUs
Left: raw linear (finite-masked for display scaling). Right: display-stretched (DISPLAY ONLY).

In [ ]:
random.seed(42)
sample_ids = [r["sample_id"] for r in manifest]
gallery_ids = random.sample(sample_ids, 6)
print("gallery:", gallery_ids)

for sid in gallery_ids:
    row = by_id[sid]
    path = RAW / row["image_path"]
    with fits.open(path, memmap=False) as hdul:
        fig, axes = plt.subplots(2, 5, figsize=(15, 6))
        fig.suptitle(f"sample {sid}")
        for i in range(5):
            data = np.asarray(hdul[i].data, dtype=np.float64)
            finite = data[np.isfinite(data)]
            vmin = np.percentile(finite, 1) if finite.size else 0
            vmax = np.percentile(finite, 99) if finite.size else 1
            axes[0, i].imshow(data, origin="lower", cmap="gray", vmin=vmin, vmax=vmax)
            axes[0, i].set_title(f"HDU {i} raw")
            axes[0, i].axis("off")
            axes[1, i].imshow(display_stretch(data), origin="lower", cmap="magma")
            axes[1, i].set_title(f"HDU {i} display")
            axes[1, i].axis("off")
        plt.tight_layout()
        plt.show()

## 3 samples containing NaNs (NaN locations shown)

In [ ]:
# samples with NaNs (from image_stats/manifest)
nan_samples = ["1-101266", "1-103458", "1-103045"]
for sid in nan_samples:
    row = by_id[sid]
    path = RAW / row["image_path"]
    with fits.open(path, memmap=False) as hdul:
        fig, axes = plt.subplots(2, 5, figsize=(15, 6))
        fig.suptitle(f"sample {sid} (NaNs highlighted)")
        for i in range(5):
            data = np.asarray(hdul[i].data, dtype=np.float64)
            nan_mask = np.isnan(data)
            axes[0, i].imshow(display_stretch(data), origin="lower", cmap="magma")
            axes[0, i].set_title(f"HDU {i} display")
            axes[0, i].axis("off")
            axes[1, i].imshow(nan_mask, origin="lower", cmap="Reds", vmin=0, vmax=1)
            axes[1, i].set_title(f"NaN n={int(nan_mask.sum())}")
            axes[1, i].axis("off")
        plt.tight_layout()
        plt.show()

## 3 samples with mask shown next to image (raw mask numeric values, separate palette)
Mask values are shown as raw numeric values with a discrete display palette. No class names are assigned.

In [ ]:
from matplotlib.colors import BoundaryNorm, ListedColormap

mask_ids = ["1-100523", "1-100725", "1-101055"]
# discrete palette for raw numeric values (display only)
raw_vals = [0, 1, 2, 3, 4]
cmap = ListedColormap(["#000000", "#e41a1c", "#377eb8", "#4daf4a", "#984ea3"])
norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5, 4.5], cmap.N)

for sid in mask_ids:
    row = by_id[sid]
    img_path = RAW / row["image_path"]
    msk_path = RAW / row["mask_path"]
    with fits.open(img_path, memmap=False) as ih, fits.open(msk_path, memmap=False) as mh:
        img = np.asarray(ih[0].data, dtype=np.float64)  # HDU 0 for display
        mask = np.asarray(mh[0].data)
        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        fig.suptitle(f"sample {sid}")
        axes[0].imshow(display_stretch(img), origin="lower", cmap="magma")
        axes[0].set_title("HDU 0 (display)")
        axes[0].axis("off")
        im = axes[1].imshow(mask, origin="lower", cmap=cmap, norm=norm)
        axes[1].set_title("mask raw values")
        axes[1].axis("off")
        plt.colorbar(im, ax=axes[1], ticks=raw_vals, fraction=0.046)
        axes[2].imshow(display_stretch(img), origin="lower", cmap="gray")
        axes[2].imshow(mask, origin="lower", cmap=cmap, norm=norm, alpha=0.35)
        axes[2].set_title("overlay (display)")
        axes[2].axis("off")
        plt.tight_layout()
        plt.show()

## Histograms per hdu_index (from image_stats.csv): median, sigma_mad, max

In [ ]:
import pandas as pd

stats = pd.read_csv(STATS)
for col, use_log in [("median", False), ("sigma_mad", False), ("max", True)]:
    fig, axes = plt.subplots(1, 5, figsize=(16, 3), sharey=False)
    fig.suptitle(f"{col} by hdu_index" + (" (log x)" if use_log else ""))
    for i in range(5):
        vals = pd.to_numeric(stats.loc[stats["hdu_index"] == i, col], errors="coerce").dropna()
        ax = axes[i]
        if use_log:
            vpos = vals[vals > 0]
            if len(vpos):
                ax.hist(vpos, bins=50)
                ax.set_xscale("log")
        else:
            ax.hist(vals, bins=50)
        ax.set_title(f"HDU {i}")
    plt.tight_layout()
    plt.show()

## Short summary (observed)
- 2,908 image files, 5 HDUs each, all 208x208, on-disk dtype `>f4` (BITPIX -32).
- Header keys `FPA.FILTER`/`FPA.FILTERID` are constant per HDU index across all images: HDU0=`g.00000`, HDU1=`r.00000`, HDU2=`i.00000`, HDU3=`z.00000`, HDU4=`y.00000` (consistent with g,r,i,z,y order; confirmed by headers).
- 127 image files contain any non-finite pixel; 162 HDUs contain NaN; 58,044 total NaN pixels; 0 ±Inf pixels; 0 all-NaN/constant/all-zero HDUs.
- NaNs occur in all HDU indices (HDU0: 78 HDUs, HDU1: 39, HDU2: 20, HDU3: 16, HDU4: 9). By distance-to-nearest-border, NaNs are mostly not on the immediate edge (distance 0: ~1.3–3.9%; distance ≤1: ~2.6–7.9%), with the majority at larger distances (scattered/interior in aggregate).
- Per-HDU finite-pixel distributions (median, sigma_mad, max, fraction_nonpositive) vary by HDU index; see histograms and `image_audit_summary.json`.
- Display stretching above is for visualization only, not training preprocessing.